## 结构化剪枝

note:
- pretranined 文件夹保存了训练好的模型参数
- datasets 文件夹保存了 cifar10 ， cifar100 两个图像分类数据集
- 需要安装 pytorch,thop,time,copy

In [ ]:
import torch
import utils
from thop import profile
import time
import resnet

def get_model_info(model,val_loader,device):#val是测试集数据加载器
    # 评估精度
    top1 = utils.get_model_accuarcy(model, val_loader, device)#这个库函数会跑完测试集
    print('   Top1 Accuracy = {:.2f}%'.format(top1))
    # 评估运行成本
    input = torch.randn(1, 3, 32, 32).to(device)
    flops, params = profile(model, inputs=(input, ), verbose=False)
    print('   FLOPs = {:.4f}G'.format(flops/1000**3))
    print('   Params = {:.4f}M'.format(params/1000**2))
    # 评估推理时间（下面的测速方式更加专业）
    st = time.time()
    model(input)
    et = time.time()-st
    print('   Cost Time = {:.4f}s'.format(et))
    #  # 更专业的测速方式
    # model.eval()
    # with torch.inference_mode():
    #     # 1. 预热 10 次
    #     for _ in range(10):
    #         model(input)
    #     if device.type == 'cuda':
    #         torch.cuda.synchronize()#等待设备上的CUDA工作完成

    #     # 2. 正式测速 100 次
    #     st = time.perf_counter()
    #     for _ in range(100):
    #         model(input)
    #     if device.type == 'cuda':
    #         torch.cuda.synchronize()
    #     et = time.perf_counter() - st
        
    # print('   Avg Cost Time = {:.4f}ms'.format((et / 100) * 1000))

In [ ]:
# 加载数据集
dataset = 'cifar10'
num_classes,train_loader,val_loader = utils.get_dataset(dataset,'datasets/',batch_size=64)
# 加载完整模型
model = resnet.resnet34(num_classes=num_classes)
model.load_state_dict(torch.load('pretrained/cifar10_resnet34.pth', map_location='cpu')['state_dict'])
# 评估完整模型
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print('==>Info of Full Model:')
# get_model_info(model,val_loader,device)

In [ ]:
# L1-norm剪枝
import copy
def _set_module(model, submodule_key, module):#辅助函数，定位 #分别对应 宿主模型 路径 新模块对象
    tokens = submodule_key.split('.')#假如这个key是layer2.0.conv1，把它拆分成["layer2", "0", "conv1"]
    sub_tokens = tokens[:-1]#找到父模块["layer2", "0"]
    cur_mod = model
    for s in sub_tokens:
        cur_mod = getattr(cur_mod, s)#获得对象的属性#几轮循环下来就找到该模型的具体位置了
    setattr(cur_mod, tokens[-1], module)#tokes[-1]指的是最后一个元素#setattr表示修改

rate = 0.5 #剪枝率
pruned_model = copy.deepcopy(model)#没有copy剪枝的时候会把原来的模型破坏掉，copy后方便剪枝后性能评估

c = 0 
for n,m in model.named_modules():
    # 剪枝卷积层
    if isinstance(m,torch.nn.Conv2d):
        # 第一层不修剪输入通道
        if c==0:#第一层输入的原始图片不能剪去
            mm=torch.nn.Conv2d(int(m.in_channels), int(m.out_channels*rate), kernel_size=m.kernel_size, stride=m.stride, padding=m.padding, bias=False)
            c=c+1
        else:
            mm=torch.nn.Conv2d(int(m.in_channels*rate), int(m.out_channels*rate), kernel_size=m.kernel_size, stride=m.stride, padding=m.padding, bias=False)
        
        # 计算每个filter的L1-norm
        l1_filter=torch.zeros((m.weight.data.size(0)))#创建记录Filter的Tensor
        for i in range(m.weight.data.size(0)):#逐条计算
            l1_filter[i]=m.weight.data[i].abs().sum()#绝对值求和
        _,l1_filter_indices = torch.sort(l1_filter, descending=True)#对Filter排序#_,表示有返回值但是丢掉
        mask_filter=torch.zeros((m.weight.data.size(0)), dtype=torch.bool)
        mask_filter[l1_filter_indices[0:int(m.weight.data.size(0)*rate)]]=True#这个0 ： size*rate其实就是保留率，保留最大的rate%
        weight_filter_temp=m.weight.data[mask_filter].detach()#提取保存的Filter   #.detach把取出来的数据单独放置

        weight_filter=torch.zeros((mm.out_channels,mm.in_channels,mm.kernel_size[0],mm.kernel_size[1]))
        for i in range(mm.out_channels):
            wt=weight_filter_temp[i]
            l1t=torch.zeros((wt.size(0)))
            for j in range(wt.size(0)):
                l1t[j]=wt[j].abs().sum()
            _,l1ti = torch.sort(l1t, descending=True)
            mask=torch.zeros((l1ti.size(0)), dtype=torch.bool)
            mask[l1ti[0:mm.in_channels]]=True
            mm.weight.data[i]=wt[mask]
        # 替换模块
        _set_module(pruned_model, n, mm)
    if isinstance(m,torch.nn.BatchNorm2d):#批量归一化#判断只有BN层才能归一化
        mm=torch.nn.BatchNorm2d(int(m.num_features*rate))
        cc=0
        for i in range(int(m.num_features)):
            if mask_filter[i]!=0:#只有被上面保留的通道才能归一化
                mm.running_mean.data[cc]=m.running_mean.data[i]#历史滑动均值
                mm.running_var.data[cc]=m.running_var.data[i]#历史滑动方差
                cc=cc+1
        _set_module(pruned_model, n, mm)#扔掉旧的BN，返回新的
#####################但是没有把新建的weight 与 baise复制过去，导致白学习了！！！！！！！！
# #########################下面是优化代码
# import torch
# from torch import nn

# def prune_bn(old_bn, keep_idx):#这个keep_idx更加高级，不用for循环一个个判断
#     new_bn = nn.BatchNorm2d(#创建新的BN
#         len(keep_idx),
#         eps=old_bn.eps,
#         momentum=old_bn.momentum,
#         affine=old_bn.affine,
#         track_running_stats=old_bn.track_running_stats
#     ).to(#把新的BN搬回去
#         device=old_bn.weight.device,
#         dtype=old_bn.weight.dtype
#     )

#     with torch.no_grad():#禁止求导
#         new_bn.weight.copy_(old_bn.weight[keep_idx])
#         new_bn.bias.copy_(old_bn.bias[keep_idx])

#         new_bn.running_mean.copy_(
#             old_bn.running_mean[keep_idx]
#         )
#         new_bn.running_var.copy_(
#             old_bn.running_var[keep_idx]
#         )

#     return new_bn
# ###############################

    if isinstance(m,torch.nn.Linear):
        mm=torch.nn.Linear(int(m.in_features*rate),m.out_features)
        mm.weight.data=m.weight.data[:,mask]
        _set_module(pruned_model, n, mm)

pruned_model=pruned_model.to(device)

In [ ]:
# 微调模型
import torch.optim as optim
import torch.nn.functional as F

print('==>Finetruning')
            
epochs=160
lr=0.1
optimizer = optim.SGD(pruned_model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)

for epoch in range(160):
    if epoch in [epochs*0.5, epochs*0.75]:
        for param_group in optimizer.param_groups:
            param_group['lr'] *= 0.1
    pruned_model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = pruned_model(data)
        loss = F.cross_entropy(output, target)
        pred = output.data.max(1, keepdim=True)[1]
        loss.backward()
        optimizer.step()
    top1 = utils.get_model_accuarcy(pruned_model, val_loader, device)
    print('   Epoch {}: Top1 Accuracy = {:.2f}%'.format(epoch,top1))
    

In [ ]:
# 评估剪枝模型
print('==>Info of UnStructured Pruned Model:')
get_model_info(pruned_model,val_loader,device)